# LLM API Usage Log Analysis

Exploring LLM API logs: token efficiency across languages, the effect of temperature on hallucination score, and why questions needing internal data hallucinate more.

_Datasets are not included in this repository._

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

df = pd.read_csv('llm_api_logs.csv')
df.head()

,log_id,language,char_length,prompt_tokens,temperature,requires_internal_data,hallucination_score
0,LOG_0001,ID,184,78,1.0,0,0.241
1,LOG_0002,EN,148,42,0.7,1,0.698
2,LOG_0003,EN,425,111,1.0,0,0.346
3,LOG_0004,ID,276,123,0.4,1,0.659
4,LOG_0005,ID,486,243,0.4,0,0.115


In [7]:
df["chars_per_token"] = df["char_length"] / df["prompt_tokens"]
df.head()

,log_id,language,char_length,prompt_tokens,temperature,requires_internal_data,hallucination_score,chars_per_token
0,LOG_0001,ID,184,78,1.0,0,0.241,2.358974
1,LOG_0002,EN,148,42,0.7,1,0.698,3.523810
2,LOG_0003,EN,425,111,1.0,0,0.346,3.828829
3,LOG_0004,ID,276,123,0.4,1,0.659,2.243902
4,LOG_0005,ID,486,243,0.4,0,0.115,2.000000


In [4]:
df.groupby("language")["chars_per_token"].mean()

,chars_per_token
language,
EN,4.017368
ID,2.147195


Bahasa Inggris lebih banyak hampir 2x dalam penggunaan karakter. Namun, dalam algoritma tokenisasi, sebagian porsi dari data tersebut berupa bahasa inggris, jadi inggris lebih hemat dan efisien saat kata-kata di encode jadi token. Satu kata bahasa Indonesia bisa dipotong-potong menjadi 2-3 token terpisah. Sehingga jika di kalkukasi, rata-rata token pada bahasa Indonesia jika dikalikan dengan 2 atau 3 bisa sama atau lebih banyak token daripada bahasa Inggris. Jadi, bahasa Indonesia lebih boros token.



In [9]:
df.groupby("temperature")["hallucination_score"].mean()

,hallucination_score
temperature,
0.1,0.174703
0.4,0.251826
0.7,0.329933
1.0,0.394381
1.5,0.513134


Semakin tinggi temperature, semakin tinggi kemungkinan model untuk berhalusinasi. Hal ini disebabkan oleh probabilitas dalam setiap token yang akan di output. Semakin tinggi temperature, maka semakin, semakin distribusi probabilitas semua token output rata. Untuk akurasi mutlak, saya akan memilih temperature paling rendah yaitu 0.1 dikarenakan model akan mengoutput token yang paling deterministik dan faktual.

In [13]:
df_internal = df[df["requires_internal_data"]==1]
df_internal

,log_id,language,char_length,prompt_tokens,temperature,requires_internal_data,hallucination_score,chars_per_token
1,LOG_0002,EN,148,42,0.7,1,0.698,3.523810
3,LOG_0004,ID,276,123,0.4,1,0.659,2.243902
5,LOG_0006,ID,488,200,0.1,1,0.561,2.440000
6,LOG_0007,ID,317,141,1.5,1,0.742,2.248227
7,LOG_0008,EN,288,80,1.0,1,0.849,3.600000
...,...,...,...,...,...,...,...,...
1987,LOG_1988,ID,258,117,0.7,1,0.657,2.205128
1993,LOG_1994,ID,295,120,0.7,1,0.630,2.458333
1994,LOG_1995,EN,189,45,0.1,1,0.550,4.200000
1996,LOG_1997,EN,467,131,1.5,1,0.969,3.564885


In [14]:
df_not = df[df["requires_internal_data"]==0]
df_not

,log_id,language,char_length,prompt_tokens,temperature,requires_internal_data,hallucination_score,chars_per_token
0,LOG_0001,ID,184,78,1.0,0,0.241,2.358974
2,LOG_0003,EN,425,111,1.0,0,0.346,3.828829
4,LOG_0005,ID,486,243,0.4,0,0.115,2.000000
8,LOG_0009,EN,445,100,0.1,0,0.000,4.450000
9,LOG_0010,EN,329,73,1.5,0,0.440,4.506849
...,...,...,...,...,...,...,...,...
1991,LOG_1992,ID,425,234,0.4,0,0.069,1.816239
1992,LOG_1993,EN,433,102,1.0,0,0.269,4.245098
1995,LOG_1996,EN,210,53,1.5,0,0.365,3.962264
1998,LOG_1999,ID,244,114,0.1,0,0.018,2.140351


In [15]:
df_internal["hallucination_score"].mean()

np.float64(0.6834212328767124)

In [16]:
df_not["hallucination_score"].mean()

np.float64(0.18927330508474574)

In [18]:
abs(df_internal["hallucination_score"].mean() - df_not["hallucination_score"].mean())

np.float64(0.4941479277919667)

Karena tanpa internal data, model akan ngasal menjawab dikarenakan jawabannya tidak ada dalam training datanya. Solusinya adalah dengan menggunakan RAG agar model tersebut dapat mengambil jawaban dari sebuah data eksternal.